# BadODD WP1: Data Reconnaissance & Audit
This notebook audits the BadODD dataset on Kaggle/Colab to answer the 6 unknowns in §5.1 and verify Gate A feasibility.

In [ ]:
import os
import glob
import zipfile
from collections import Counter, defaultdict
import pandas as pd
from PIL import Image

# Auto-detect BadODD folder in /kaggle/input
print("Scanning /kaggle/input for BadODD data...")
input_dirs = glob.glob('/kaggle/input/**/images', recursive=True)
if input_dirs:
    DATA_DIR = os.path.dirname(input_dirs[0])
else:
    DATA_DIR = '/kaggle/input/dl-enigma-10-sust-cse-carnival-2024/dlenigma1/BadODD'

print('Detected DATA_DIR:', DATA_DIR)
os.system(f'ls -lh "{DATA_DIR}"')

In [ ]:
DISTRICTS = ['chittagong', 'dhaka', 'sylhet', 'rajshahi', 'mymensingh', 'maowa', 'sirajganj', 'sherpur', 'khulna']

def get_district(filename):
    f = os.path.basename(filename).lower()
    for d in DISTRICTS:
        if d in f:
            return d
    return 'unknown'

# Find all images and labels
all_images = glob.glob(os.path.join(DATA_DIR, '**', '*.jpg'), recursive=True) + \
             glob.glob(os.path.join(DATA_DIR, '**', '*.png'), recursive=True)
all_images = [f for f in all_images if not os.path.basename(f).startswith('._')]

train_images = [f for f in all_images if 'train' in f.lower()]
test_images = [f for f in all_images if 'test' in f.lower()]

print(f'Total images found: {len(all_images)}')
print(f'Train images: {len(train_images)}')
print(f'Test images: {len(test_images)}')

district_all = Counter([get_district(f) for f in all_images])
district_train = Counter([get_district(f) for f in train_images])

print('\n--- District Image Distribution (All Images) ---')
for d, c in district_all.most_common():
    print(f'{d:15s}: {c}')

print('\n--- District Image Distribution (Train / Labeled) ---')
for d, c in district_train.most_common():
    print(f'{d:15s}: {c}')

In [ ]:
CLASSES = [
    'auto_rickshaw', 'bicycle', 'bus', 'car', 'cart_vehicle', 
    'construction_vehicle', 'motorbike', 'person', 'priority_vehicle', 
    'three_wheeler', 'train', 'truck', 'wheelchair'
]

label_files = glob.glob(os.path.join(DATA_DIR, '**', 'labels', '**', '*.txt'), recursive=True)
label_files = [f for f in label_files if not os.path.basename(f).startswith('._')]
print(f'Total label files found: {len(label_files)}')

counts_per_district = defaultdict(Counter)
crops_saved = defaultdict(int)
os.makedirs('audit_crops', exist_ok=True)

out_of_bounds = 0
total_boxes = 0

for lf in label_files:
    dist = get_district(lf)
    base_name = os.path.splitext(os.path.basename(lf))[0]
    
    # Find matching image file
    img_path = None
    for ext in ['.jpg', '.jpeg', '.png']:
        candidate = lf.replace('/labels/', '/images/').replace('.txt', ext)
        if os.path.exists(candidate):
            img_path = candidate
            break
            
    with open(lf, 'r') as f:
        for line in f:
            parts = line.strip().split()
            if len(parts) >= 5:
                cid = int(parts[0])
                cx, cy, w, h = map(float, parts[1:5])
                cname = CLASSES[cid] if cid < len(CLASSES) else f'class_{cid}'
                counts_per_district[dist][cname] += 1
                total_boxes += 1
                
                if cx < 0 or cx > 1 or cy < 0 or cy > 1 or w <= 0 or h <= 0:
                    out_of_bounds += 1
                    
                # Save crops for ambiguous classes
                if img_path and cname in ['three_wheeler', 'auto_rickshaw', 'cart_vehicle', 'wheelchair'] and crops_saved[cname] < 5:
                    try:
                        with Image.open(img_path) as img:
                            iw, ih = img.size
                            x1 = max(0, int((cx - w/2) * iw))
                            y1 = max(0, int((cy - h/2) * ih))
                            x2 = min(iw, int((cx + w/2) * iw))
                            y2 = min(ih, int((cy + h/2) * ih))
                            if x2 > x1 and y2 > y1:
                                crop = img.crop((x1, y1, x2, y2))
                                crop.save(f'audit_crops/{cname}_{crops_saved[cname]}.jpg')
                                crops_saved[cname] += 1
                    except Exception as e:
                        pass

df_counts = pd.DataFrame(counts_per_district).fillna(0).astype(int)
df_counts['Total'] = df_counts.sum(axis=1)
print('\n=== INSTANCE COUNTS TABLE ===')
print(df_counts[['dhaka', 'chittagong', 'Total']])

df_counts.to_csv('district_class_counts.csv')
print(f'Total bounding boxes: {total_boxes}, Out-of-bounds: {out_of_bounds}')

In [ ]:
# Gate A Feasibility Summary
ctg_img_count = district_train.get('chittagong', 0)
ctg_classes = df_counts['chittagong'] if 'chittagong' in df_counts else pd.Series()

print('========================================')
print('          GATE A EVALUATION             ')
print('========================================')
print(f'Chattogram Labeled Images: {ctg_img_count} (Required: >= 300)')
if ctg_img_count >= 300:
    print('  -> Condition 1: PASSED')
else:
    print('  -> Condition 1: FAILED / NEEDS FALLBACK')

print('\nPer-Class Chattogram Counts (Required: >= 30 for study classes):')
for cname in CLASSES:
    cnt = ctg_classes.get(cname, 0)
    status = 'PASS' if cnt >= 30 else 'WARN (< 30)'
    print(f'  - {cname:22s}: {cnt:5d}  [{status}]')

# Zip audit crops for easy download
os.system('zip -r audit_crops.zip audit_crops/ district_class_counts.csv')